# BSIL Atlas — Pipeline Coverage Report

Connects to the draft pipeline tables and reports on:
1. Care offering breakdown by type
2. Source coverage by LA — which LAs have multi-source data?
3. Source reliance — how do sources flow into care types?
4. Field provenance — where do resolved names/postcodes actually come from?

In [ ]:
import os

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import psycopg

conn = psycopg.connect(
    host=os.environ.get("POSTGRES_HOST", "postgres"),
    port=os.environ.get("POSTGRES_PORT", 5432),
    user=os.environ.get("POSTGRES_USER", "bsil"),
    password=os.environ.get("POSTGRES_PASSWORD", "bsil_local"),
    dbname=os.environ.get("POSTGRES_DB", "bsil"),
)
conn.autocommit = True
print("Connected")

## 1. Provider breakdown

In [ ]:
# Headline counts
totals = pd.read_sql("""
SELECT
    COUNT(*) AS total,
    COUNT(*) FILTER (WHERE excluded) AS excluded,
    COUNT(*) FILTER (WHERE redacted_childminder) AS redacted,
    COUNT(*) FILTER (WHERE NOT excluded AND NOT redacted_childminder AND lad25cd NOT LIKE 'E%%') AS non_english,
    COUNT(*) FILTER (WHERE NOT excluded AND NOT redacted_childminder AND lad25cd LIKE 'E%%') AS analysed
FROM draft.providers
""", conn)
total = int(totals["total"].iloc[0])
excluded = int(totals["excluded"].iloc[0])
redacted = int(totals["redacted"].iloc[0])
non_english = int(totals["non_english"].iloc[0])
analysed = int(totals["analysed"].iloc[0])
print(f"Total providers: {total:,}")
print(f"  Excluded (not childcare): {excluded:,}")
print(f"  Redacted childminders:    {redacted:,}")
print(f"  Non-English LAs:          {non_english:,}")
print(f"  Analysed (England):       {analysed:,}")

# Institution type breakdown for analysed providers
inst = pd.read_sql("""
SELECT institution_type, COUNT(*) AS n
FROM draft.providers
WHERE NOT excluded AND NOT redacted_childminder AND lad25cd LIKE 'E%%'
GROUP BY 1
ORDER BY n DESC
""", conn)

# Care type breakdown by institution type (outer ring)
# Uses unnest so providers with multiple care_types appear in each.
care_by_inst = pd.read_sql("""
WITH care_expanded AS (
    SELECT institution_type, unnest(care_types) AS care_type, COUNT(*) AS n
    FROM draft.providers
    WHERE NOT excluded AND NOT redacted_childminder AND lad25cd LIKE 'E%%'
    AND array_length(care_types, 1) > 0
    GROUP BY 1, 2
),
no_care AS (
    SELECT institution_type, 'no care type' AS care_type, COUNT(*) AS n
    FROM draft.providers
    WHERE NOT excluded AND NOT redacted_childminder AND lad25cd LIKE 'E%%'
    AND (care_types = '{}' OR care_types IS NULL)
    GROUP BY 1
)
SELECT * FROM care_expanded
UNION ALL
SELECT * FROM no_care
ORDER BY 1, 3 DESC
""", conn)

# Sum of care_type counts per institution_type (>= provider count due to multi-type)
inst_care_sums = care_by_inst.groupby("institution_type")["n"].sum().to_dict()

# Three-ring sunburst using ids to avoid label collisions.
# The outer ring counts provider-offerings (unnested), so middle-ring values
# must match those sums for branchvalues="total" to work.
# The inner-ring "England" value is the sum of all institution_type sums,
# and the root is that plus excluded + redacted + non_english.

england_offerings = sum(inst_care_sums.values())
root_total = excluded + redacted + non_english + england_offerings

ids =     ["root",          "excluded",  "redacted",              "non_english",   "england"]
labels =  ["All providers", "Excluded",  "Redacted childminder", "Non-English LA","England"]
parents = ["",              "root",      "root",                  "root",          "root"]
values =  [root_total,      excluded,    redacted,                non_english,     england_offerings]

for _, row in inst.iterrows():
    itype = row["institution_type"]
    ids.append(f"inst_{itype}")
    labels.append(itype)
    parents.append("england")
    values.append(inst_care_sums.get(itype, int(row["n"])))

for _, row in care_by_inst.iterrows():
    itype = row["institution_type"]
    ctype = row["care_type"]
    ids.append(f"care_{itype}_{ctype}")
    labels.append(ctype)
    parents.append(f"inst_{itype}")
    values.append(int(row["n"]))

fig = go.Figure(go.Sunburst(
    ids=ids,
    labels=labels,
    parents=parents,
    values=values,
    branchvalues="total",
    textinfo="label+value",
    insidetextorientation="radial",
    maxdepth=4,
))
fig.update_layout(
    title=f"Provider breakdown ({total:,} providers — outer ring shows care offerings)",
    height=700,
    margin={"t": 50, "b": 10, "l": 10, "r": 10},
)
fig.show()

## 2. Source coverage by LA

In [ ]:
import geopandas as gpd

# OS Boundary-Line GeoPackage — mounted from source_data/
GPKG_PATH = "/opt/source_data/boundary-line/Data/bdline_gb.gpkg"

gdf = gpd.read_file(GPKG_PATH, layer="district_borough_unitary")
gdf = gdf.rename(columns={"Census_Code": "LAD25CD", "Name": "LAD25NM"})

# Simplify while still in BNG (metres) — 200m tolerance is fine for national maps
gdf["geometry"] = gdf.geometry.simplify(tolerance=200, preserve_topology=True)
gdf = gdf.to_crs(epsg=4326)

la_sources = pd.read_sql("""
SELECT lad25cd,
       bool_or('la_scrape' = ANY(sources)) AS has_la_scrape,
       bool_or('ofsted' = ANY(sources)) AS has_ofsted,
       bool_or('school_census' = ANY(sources)) AS has_school,
       bool_or('free_breakfast' = ANY(sources)) AS has_free_breakfast,
       COUNT(*) AS providers
FROM draft.providers
WHERE lad25cd IS NOT NULL AND NOT excluded AND NOT redacted_childminder
GROUP BY lad25cd
""", conn)

la_sources["source_tier"] = la_sources.apply(
    lambda r: "LA + Ofsted + School" if r.has_la_scrape and r.has_ofsted and r.has_school
    else "LA + Ofsted" if r.has_la_scrape and r.has_ofsted
    else "LA only" if r.has_la_scrape
    else "Ofsted + School" if r.has_ofsted and r.has_school
    else "Ofsted only" if r.has_ofsted
    else "Other",
    axis=1,
)

tier_merged = gdf.merge(
    la_sources[["lad25cd", "source_tier", "providers"]],
    left_on="LAD25CD", right_on="lad25cd", how="left",
)
tier_merged["source_tier"] = tier_merged["source_tier"].fillna("No data")

fig = px.choropleth_map(
    tier_merged,
    geojson=tier_merged.geometry.__geo_interface__,
    locations=tier_merged.index,
    color="source_tier",
    hover_name="LAD25NM",
    hover_data={"providers": True, "lad25cd": True},
    color_discrete_map={
        "LA + Ofsted + School": "#2ca02c",
        "LA + Ofsted": "#98df8a",
        "LA only": "#ffbb78",
        "Ofsted + School": "#aec7e8",
        "Ofsted only": "#ff9896",
        "Other": "#c7c7c7",
        "No data": "#d62728",
    },
    title="Source coverage tier by Local Authority",
    center=dict(lat=54.5, lon=-2.5),
    zoom=4.8,
    map_style="white-bg",
)
fig.update_layout(
    height=900,
    margin={"l": 0, "r": 0, "t": 40, "b": 0},
    legend=dict(title_text="Source tier"),
)
fig.show(config={"scrollZoom": False})

## 3. Source reliance — Sankey diagram

Shows how care offerings flow from source to care_type to provider count.

In [ ]:
flows = pd.read_sql("""
SELECT co.source, co.care_type, COUNT(DISTINCT ps.provider_id) AS providers
FROM draft.care_offerings co
JOIN draft.provider_sources ps ON ps.care_offering_id = co.id
JOIN draft.providers p ON ps.provider_id = p.provider_id
WHERE co.care_type IS NOT NULL AND NOT p.excluded AND NOT p.redacted_childminder
GROUP BY co.source, co.care_type
ORDER BY providers DESC
""", conn)

sources = flows["source"].unique().tolist()
care_types = flows["care_type"].unique().tolist()
labels = sources + care_types

fig = go.Figure(go.Sankey(
    node=dict(label=labels, pad=15, thickness=20),
    link=dict(
        source=[sources.index(s) for s in flows["source"]],
        target=[len(sources) + care_types.index(ct) for ct in flows["care_type"]],
        value=flows["providers"].tolist(),
    ),
))
fig.update_layout(title="Source -> Care Type (provider count)", height=600)
fig.show()

## 4. Field provenance — where do resolved fields come from?

In [ ]:
provenance = pd.read_sql("""
SELECT
    metadata->'field_sources'->>'provider_name' AS name_source,
    metadata->'field_sources'->>'postcode' AS postcode_source,
    metadata->'field_sources'->>'lad25cd' AS lad25cd_source,
    COUNT(*) AS n
FROM draft.providers
WHERE metadata <> '{}'::jsonb
  AND NOT excluded AND NOT redacted_childminder
GROUP BY 1, 2, 3
ORDER BY n DESC
""", conn)
provenance

In [ ]:
# Stacked bar: for each resolved field, which source won?
frames = []
for field in ["provider_name", "postcode", "lad25cd"]:
    df = pd.read_sql(f"""
        SELECT metadata->'field_sources'->>'{field}' AS source, COUNT(*) AS n
        FROM draft.providers
        WHERE metadata->'field_sources'->>'{field}' IS NOT NULL
          AND NOT excluded AND NOT redacted_childminder
        GROUP BY 1
    """, conn)
    df["field"] = field
    frames.append(df)

prov_df = pd.concat(frames)

fig = px.bar(
    prov_df, x="field", y="n", color="source",
    title="Field provenance: which source provided each resolved field?",
    labels={"n": "providers", "field": "", "source": "winning source"},
    barmode="stack",
)
fig.update_layout(height=450)
fig.show()

In [ ]:
conn.close()
print("Done")